## Tearing sensitivities in $\eta$ and $k$: running $S$-exponents along $\gamma(k)$, and $k_{max}$ by Newton

**Rung 2 (science) of `plans/AUTODIFF_PLAN.md`.** Rung 1b (`tearing-sensitivity.ipynb`) differentiated the
tearing eigenvalue along directions of the *equilibrium* ($\alpha$, the sheet width). Rung 2's overrides seam
(`taranis/overrides.py`) makes the *physics parameters and the wavenumber* differentiable too: the kgrid is
rebuilt by `setup_kgrids(params, overrides=...)` from traced values, so
$$\frac{d\lambda}{dp}=\frac{w^H(\partial J/\partial p)\,v}{w^Hv}$$
with $\partial J/\partial p$ a forward-over-forward `jax.jvp` through the kgrid build and the RHS
(`stability.dj_operator(..., dparams={key: tangent}, overrides=...)`). Here:

* $\partial_\eta$: `dparams={"diss": (0, 1)}` -- `diss` is the $(\nu,\eta)$ pair, so this tangent moves $\eta$ **only**
  ($\nu=0$ stays 0). Local exponent $q_S=d\ln\gamma/d\ln S=-(\eta/\gamma)\,\partial_\eta\gamma$.
* $\partial_k$: `dparams={"Ly": -Ly/k}` -- a mode block is an index, $k_y=2\pi/L_y$, so $dL_y/dk=-L_y/k$.
  $q_k=d\ln\gamma/d\ln k=(k/\gamma)\partial_k\gamma$.
* $\partial_\alpha$ along $dx_0=(\Psi_0,0)$ as in rung 1b.

**The wavenumber is itself an override**: every `Parameters` has the placeholder $L_y=2\pi$ and the block's $k$ is
`overrides={"Ly": 2*pi/k}` -- one compile serves every $k$, and a Newton solve moves $k$ with the box $(L_x,n_x)$ held fixed.

**Setup** (rungs 1b-ref/1b): `dims=2`, $\nu=0$, `hyper=1`, $a=v_{Ay}=1$, $S=1/\eta$, $\Psi_0=\mathrm{sech}^2x$ (+2 images),
$\Phi_0=\alpha\Psi_0$, ky block, fp64. Data: `tearing_rung2_run.py` (every solve one cache file under
`examples/data/tearing-rung2/`); this notebook only reads.

**Which scalings.** For $\Delta'a\propto(ka)^{-n}$ the FKR/Coppi crossing gives $k_{max}\propto S^{-1/(3n+1)}$,
$\gamma_{max}\propto S^{-(n+1)/(3n+1)}$: this sheet has $n=2$, so **$-1/7$ and $-3/7$ already at $\alpha=0$** (rung 1a;
the familiar $-1/4$, $-1/2$ are $n=1$). The paper's eqs. 5.13-5.14 / 8.1 (Mallet, Eriksson, Swisdak & Juno, JPP 2025) give
the SAME $S$-exponents in the shear-modified regime, $k_{tr}a\sim(\alpha/(1-\alpha^2))^{1/7}S^{-1/7}$ (the crossing of the
shear constant-$\psi$ $\gamma\sim\alpha^{1/2}(1-\alpha^2)^{1/2}S^{-1/2}(ka)^{-1/2}$ and nonconstant-$\psi$
$\gamma\sim\alpha^{1/3}(1-\alpha^2)^{2/3}(ka)^{2/3}S^{-1/3}$) and $\gamma_{max}\sim\alpha^{3/7}(1-\alpha^2)^{4/7}S^{-3/7}$, valid for
$1-\alpha^2\ll\alpha$ with a thin layer $\delta_{in}/a=(\alpha/(1-\alpha^2))^{1/3}(kaS)^{-1/3}\ll1$ (eq. 5.15) -- its fig. 4
is at $1-\alpha^2=0.01$, $S=10^8$-$10^{16}$. So the $S$-exponents do **not** discriminate the regimes; what does is the
$\alpha$-dependence and, at fixed $k$, $q_S$: $-3/5$ (FKR) / $-1/3$ (Coppi) without shear, $-1/2$ / $-1/3$ with it.

In [ ]:
# fp64 before taranis/jax import (TARANIS_PRECISION is read at import); the profile picks the cache
import os
os.environ["TARANIS_PRECISION"] = "64"
os.environ.setdefault("TARANIS_R2_PROFILE", "full")
import time
import numpy as np
import matplotlib.pyplot as plt
import tearing_eigen_run as T
import tearing_sensitivity_run as U
import tearing_rung2_run as R

t0 = time.time()
FD = R.fd_results()
DISP = {(S, al): R.disp_results(S, al) for S in R.S_DISP for al in R.ALPHA_DISP}
KA = {S: R.kmax_alpha_results(S) for S in R.S_KA}
KS = {al: R.kmax_S_results(al, hi=(al == 0.0)) for al in R.ALPHA_KS}
G2 = R.grid2_results()
print(f"profile {R.PROFILE}: read {sum(len(v.get(1, {})) for v in DISP.values())} dispersion points, "
      f"{sum(len(v) for v in KA.values())} k_max(alpha) and {sum(len(v) for v in KS.values())} k_max(S) "
      f"solutions, {len(FD)} FD points, {len(G2)} 2nx checks in {time.time() - t0:.1f} s")

### 1. Checks: the seam, the theory's analytic exponents, the FD gates

(a) $J$ at `overrides={"Ly": 2π/k}` against $J$ of a `Parameters` built with that $L_y$ (the rung-2 gate, re-checked on this
setup). (b) The exact $\alpha=0$ dispersion relation $\Delta'\delta_\eta=F(\Lambda)$ (rung 1a) differentiated
**analytically**: with $m=d\ln F/d\ln\Lambda=\tfrac54+\tfrac38 b[\psi(\tfrac{b-1}4)-\psi(\tfrac{b+5}4)]$, $b=\Lambda^{3/2}$,
$$q_S=-\tfrac13-\tfrac1{3m},\qquad q_k=\tfrac23+\frac{d\ln\Delta'/d\ln k-\tfrac13}{m},$$
against central differences of $\gamma_{th}$. (c) The sensitivities against central differences of eigenvalues, in $\eta$
(`overrides` moving $\eta$ in the pair) and in $k$, at the sensitivity's own grid.

In [ ]:
import taranis as jr
from taranis import stability as st
S_, k_, al_, nx_, Lx_ = 1e4, 0.5, 0.8, 512, 30.0
p = R.make_params(S_, nx_, Lx_)
x0 = U.equilibrium(p, al_)[0]
Jo, _ = st.ky_block_matrix(x0, jr.setup_kgrids(p), p, 1, overrides=R.k_override(k_))
p2 = jr.Parameters(nx=nx_, ny=8, Lx=Lx_, Ly=2*np.pi/k_, dims=2, cfl_safety=0.5, forcing=False,
                   eqpars={"diss": (0.0, 1/S_), "hyper": 1})
Jp, _ = st.ky_block_matrix(U.equilibrium(p2, al_)[0], jr.setup_kgrids(p2), p2, 1)
print(f"(a) max |J(overrides Ly) - J(Parameters Ly)| = {np.abs(Jo - Jp).max():.1e}  (max |J| = {np.abs(Jp).max():.1e})")
print("(b) theory exponents: analytic vs central difference of gamma_theory (h = 1e-4 in ln)")
h = 1e-4
for ka in (0.1, 0.3, 1.0, 2.0):
    for S in (1e3, 1e5):
        fS = (np.log(T.gamma_theory(ka, S*np.exp(h))) - np.log(T.gamma_theory(ka, S*np.exp(-h))))/(2*h)
        fk = (np.log(T.gamma_theory(ka*np.exp(h), S)) - np.log(T.gamma_theory(ka*np.exp(-h), S)))/(2*h)
        print(f"    ka={ka:4.1f} S={S:.0e}: q_S {R.theory_s_exponent(ka, S):+.9f} (FD {fS:+.9f})   "
              f"q_k {R.theory_k_exponent(ka, S):+.7f} (FD {fk:+.7f})")

In [ ]:
print("(c) FD gate: central differences of eigenvalues (relative steps h in eta and in k) vs the sensitivity")
print("    S      ka   alpha   nx   | dir |  sensitivity         |  h      |FD - sens|  rel.      | order")
FDROWS = []
for (S, ka, al), (rec, rows) in FD.items():
    for which in ("eta", "k"):
        d = complex(rec["dlam_eta" if which == "eta" else "dlam_k"]).real
        hs = np.array([r[0] for r in rows[which]]); es = np.array([r[2] for r in rows[which]])
        order = np.polyfit(np.log(hs), np.log(es), 1)[0]
        FDROWS.append((S, ka, al, which, order, es[-1]/abs(d)))
        for i, (hh, fd, e) in enumerate(rows[which]):
            head = (f"  {S:.0e}  {ka:4.2f}  {al:4.2f}  {int(rec['nx']):5d}  | {which:3s} | {d:+.12e} |"
                    if i == 0 else " "*43 + "|" + " "*22 + "|")
            print(f"{head} {hh:7.0e}  {e:.2e}  {e/abs(d):.1e}  |" + (f" {order:.3f}" if i == len(rows[which]) - 1 else ""))
    print(f"{'':43s}  gamma = {complex(rec['lam']).real:.10e}, residuals r/l {float(rec['res_r']):.1e}/"
          f"{float(rec['res_l']):.1e}, kappa {float(rec['kappa']):.2e}, rel_err {float(rec['rel_err']):.1e}")

### 2. The local $S$-exponent along the dispersion curve

$q_S(k)=-(\eta/\gamma)\partial_\eta\gamma$ at every $k$ of a sweep, one sensitivity per point (no sweep in $S$), at
$S=10^3$-$10^5$ and $\alpha=0,0.3,0.6,0.8$; rung 1a's per-$k$ box $L_x=\max(15/k,20)$, $n_x$ from its converged ladders,
every point also at $n_x/2$. At $\alpha=0$ the exact relation's analytic $q_S$ is the reference (it is an asymptotic
theory: agreement is a trend in $S$, the eigenvalue carrying rung 1a's finite-$S$ correction
$\simeq C(k)/(\gamma\tau_R)$, strongest near the marginal point $ka=\sqrt5$).

In [ ]:
def disp_curve(S, al, lv=1):
    f = DISP[(S, al)].get(lv, {})
    ka = np.array(sorted(f))
    return ka, [f[k] for k in ka]

print("grid convergence of the sweeps: nx vs nx/2 (max over ka), and the worst residual / isolation")
print("    S     alpha | points | max rel d gamma   max |d q_S|   max |d q_k|  | max res   max rel_err  max kappa")
for (S, al) in sorted(DISP):
    hi, lo = DISP[(S, al)].get(1, {}), DISP[(S, al)].get(2, {})
    com = [k for k in hi if k in lo]
    if not com:
        continue
    dg = max(abs(complex(hi[k]["lam"])/complex(lo[k]["lam"]) - 1) for k in com)
    dqs = max(abs(R.s_exponent(hi[k]) - R.s_exponent(lo[k])) for k in com)
    dqk = max(abs(R.k_exponent(hi[k]) - R.k_exponent(lo[k])) for k in com)
    res = max(max(float(hi[k]["res_r"]), float(hi[k]["res_l"])) for k in hi)
    print(f"  {S:.0e}  {al:4.2f}  |  {len(hi):3d}   |  {dg:.1e}          {dqs:.1e}       {dqk:.1e}     | {res:.1e}   "
          f"{max(float(hi[k]['rel_err']) for k in hi):.1e}      {max(float(hi[k]['kappa']) for k in hi):.1e}")
missing = [(S, al, k) for (S, al) in DISP for k in R.KA_DISP if k not in DISP[(S, al)].get(1, {})]
print(f"\nnot available (refused / failed / not computed): {missing if missing else 'none'}")

In [ ]:
print("q_S = d ln gamma/d ln S along gamma(k), alpha = 0: eigen-sensitivity | exact theory (analytic)")
print("   ka  | " + " | ".join(f"   S = {S:.0e}: eig      th   " for S in R.S_DISP))
for ka in R.KA_DISP:
    row = []
    for S in R.S_DISP:
        f = DISP[(S, 0.0)].get(1, {})
        row.append(f"{R.s_exponent(f[ka]):+.4f}  {R.theory_s_exponent(ka, S):+.4f}" if ka in f else " "*16)
    print(f"  {ka:4.2f} |   " + "   |   ".join(row))

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))
for j, S in enumerate(R.S_DISP):
    ka, recs = disp_curve(S, 0.0)
    if len(ka):
        axs[0].semilogx(ka, [R.s_exponent(r) for r in recs], "o", color=f"C{j}", ms=4, label=f"eig, S = {S:.0e}")
    kk = np.geomspace(0.05, 2.23, 200)
    axs[0].semilogx(kk, [R.theory_s_exponent(k, S) for k in kk], "-", color=f"C{j}", lw=1)
    km, _ = R.theory_kmax(S)
    axs[0].axvline(km, color=f"C{j}", ls=":", lw=0.8)
for y, lab in ((-3/5, "FKR -3/5"), (-1/3, "Coppi -1/3"), (-3/7, "-3/7 (n=2 max)")):
    axs[0].axhline(y, color="k", ls="--" if y != -3/7 else ":", lw=0.8)
    axs[0].text(0.052, y + 0.006, lab, fontsize=8)
axs[0].set_xlabel("ka"); axs[0].set_ylabel("d ln gamma / d ln S"); axs[0].set_title("alpha = 0: dots eigen-sensitivity, lines exact theory, dotted: theory k_max")
axs[0].legend(fontsize=8); axs[0].set_ylim(-0.7, 0.3)
for j, S in enumerate(R.S_DISP):
    ka, recs = disp_curve(S, 0.0)
    if len(ka):
        axs[1].semilogx(ka, [R.k_exponent(r) for r in recs], "o", color=f"C{j}", ms=4)
    kk = np.geomspace(0.05, 2.1, 200)
    axs[1].semilogx(kk, [R.theory_k_exponent(k, S) for k in kk], "-", color=f"C{j}", lw=1, label=f"theory S = {S:.0e}")
for y in (2/3, -6/5):
    axs[1].axhline(y, color="k", ls="--", lw=0.8)
axs[1].set_ylim(-4, 1); axs[1].set_xlabel("ka"); axs[1].set_ylabel("d ln gamma / d ln k"); axs[1].legend(fontsize=8)
axs[1].set_title("k-exponent (Coppi 2/3, FKR n=2 -6/5)")
plt.tight_layout()

In [ ]:
# the envelope identity at the maximum: q_S at k_max IS d ln gamma_max/d ln S. From the sweep: q_S interpolated
# (quadratic in ln k through the 3 points around the largest gamma) at the sweep's k_max, against the Newton
# solution's -(eta/gamma*) dgamma/deta at k* (section 4) and the theory's
print("   S    | sweep k_max  q_S there | Newton k*   q_S at k*  | theory k_max  q_S(k_max) | n=2 asymptote -3/7")
for S in R.S_DISP:
    ka, recs = disp_curve(S, 0.0)
    if len(ka) < 3:
        continue
    g = np.array([complex(r["lam"]).real for r in recs]); i = int(np.clip(np.argmax(g), 1, len(g) - 2))
    lk = np.log(ka[i - 1:i + 2]); cg = np.polyfit(lk, g[i - 1:i + 2], 2); lkm = -cg[1]/(2*cg[0])
    q = np.polyval(np.polyfit(lk, [R.s_exponent(r) for r in recs[i - 1:i + 2]], 2), lkm)
    nw = KA.get(S, {}).get(0.0)
    kth = R.theory_kmax(S)[0]
    print(f"  {S:.0e} |  {np.exp(lkm):.4f}     {q:+.4f}   | " +
          (f"{nw['kmax']:.6f}   {nw['slope_g_S']:+.5f}  " if nw else " "*23) +
          f"|  {kth:.6f}     {R.theory_s_exponent(kth, S):+.5f}  | {-3/7:+.5f}")

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 4.4), sharey=True)
for ax, S in zip(axs, R.S_DISP):
    for j, al in enumerate(R.ALPHA_DISP):
        ka, recs = disp_curve(S, al)
        if len(ka):
            ax.semilogx(ka, [R.s_exponent(r) for r in recs], "o-", ms=3, color=f"C{j}", label=f"alpha = {al}")
    kk = np.geomspace(0.1, 2.2, 200)
    ax.semilogx(kk, [R.theory_s_exponent(k, S) for k in kk], "k-", lw=0.7, label="theory alpha = 0")
    for y, ls in ((-3/5, "--"), (-1/2, "-."), (-1/3, "--")):
        ax.axhline(y, color="gray", ls=ls, lw=0.8)
    ax.set_title(f"S = {S:.0e}  (dashed -3/5, -1/3; dash-dot shear const-psi -1/2)"); ax.set_xlabel("ka")
    ax.set_ylim(-0.75, 0.3)
axs[0].set_ylabel("d ln gamma / d ln S"); axs[0].legend(fontsize=8)
plt.tight_layout()

In [ ]:
# where the shear regime could apply: lambda = gamma/(k delta_in) (the paper's eq. 4.7 with delta = gamma/k,
# delta_in eq. 5.1) against alpha, and Delta' delta_in -- along each sweep
print("paper's regime parameters along the sweeps (prefactor-free): Delta' delta_in and lambda/alpha "
      "(shear matters for lambda << alpha)")
for S in R.S_DISP:
    for al in R.ALPHA_DISP[1:]:
        ka, recs = disp_curve(S, al)
        if not len(ka):
            continue
        g = np.array([complex(r["lam"]).real for r in recs])
        din = R.delta_in(S, ka, al)
        lam_ = g/ka/din
        print(f"  S={S:.0e} alpha={al}: ka      " + " ".join(f"{k:6.2f}" for k in ka))
        print(f"  {'':22s}Delta'd_in " + " ".join(f"{v:6.2f}" for v in T.dprime_sech2(ka)*din))
        print(f"  {'':22s}lambda/a   " + " ".join(f"{v:6.3f}" for v in lam_/al))
        print(f"  {'':22s}q_S        " + " ".join(f"{R.s_exponent(r):+6.3f}" for r in recs))

**Result (section 2).** *To be written from the full campaign.*

### 3. $k_{max}(\alpha)$ by Newton on $\partial_k\gamma=0$ (the paper's fig. 3, bottom right)

At each $S$ one box, $L_x=20/k_{th}(S)$ ($k_{th}$ the $\alpha=0$ theory maximum) and one $n_x$ (rung 1a's converged
resolution at $1.5k_{th}$), for **every** $\alpha$ -- so $k_{max}(\alpha)$ is the maximum of one discrete operator family,
with no box or grid jumps. Newton with a secant curvature (one eigen-solve per iterate), continued in $\alpha$ with the
previous solution's implicit $dk^*/d\alpha$; then two solves at $k^*(1\pm0.02)$ give $\gamma_{kk}$, $\gamma_{k\eta}$,
$\gamma_{k\alpha}$ by central differences, hence
$$\frac{dk^*}{dp}=-\frac{\gamma_{kp}}{\gamma_{kk}},\qquad\frac{d\gamma^*}{dp}=\partial_p\gamma\big|_{k^*}\ \text{(envelope)}.$$
Grid check: the $k_{max}$ shift at $n_x/2$, $-\partial_k\gamma^{(n_x/2)}(k^*)/\gamma_{kk}$ (one solve).

In [ ]:
print("   S     alpha  1-a^2  | k*            gamma*          | its  |step|/k  |k_pol-k*|/k  grid dk/k   grid dg/g | "
      "kappa     res     | d ln k*/d ln(1-a^2)  d ln g*/d ln(1-a^2)")
for S in R.S_KA:
    for al, r in sorted(KA[S].items()):
        its = r["its"]
        print(f"  {S:.0e}  {al:5.3f}  {1-al*al:6.4f} | {r['kmax']:.10f}  {r['gmax']:.10e} | {r['iterations']:2d}   "
              f"{abs(its[-1]['step'])/r['kmax']:.0e}   {abs(r['k_polished']-r['kmax'])/r['kmax']:.0e}       "
              f"{r['grid_dk']/r['kmax']:+.1e}   {r['grid_dgamma']/r['gmax']:+.1e} | {float(r['rec']['kappa']):.2e}  "
              f"{max(float(r['rec']['res_r']), float(r['rec']['res_l'])):.0e} | {r['slope_k_u']:+.5f}             "
              f"{r['slope_g_u']:+.5f}")

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(17, 4.5))
for j, S in enumerate(R.S_KA):
    al = np.array(sorted(KA[S]))
    if not len(al) or 0.0 not in KA[S]:
        continue
    u = 1 - al**2
    k0, g0 = KA[S][0.0]["kmax"], KA[S][0.0]["gmax"]
    axs[0].loglog(u, [KA[S][a]["kmax"]/k0 for a in al], "o-", ms=3, color=f"C{j}", label=f"S = {S:.0e}")
    axs[1].loglog(u, [KA[S][a]["gmax"]/g0 for a in al], "o-", ms=3, color=f"C{j}", label=f"S = {S:.0e}")
    m = al > 0
    axs[2].semilogx(u[m], [KA[S][a]["slope_k_u"] for a in al[m]], "o-", ms=3, color=f"C{j}", label=f"k*, S = {S:.0e}")
    axs[2].semilogx(u[m], [KA[S][a]["slope_g_u"] for a in al[m]], "s--", ms=3, color=f"C{j}", label=f"gamma*, S = {S:.0e}")
uu = np.geomspace(5e-3, 1, 50)
axs[0].loglog(uu, uu**(-1/7), "k:", label="(1-a^2)^(-1/7)")
axs[1].loglog(uu, uu**(4/7), "k:", label="(1-a^2)^(4/7)")
for y in (-1/7, 4/7):
    axs[2].axhline(y, color="k", ls=":", lw=0.8)
for ax, t in zip(axs, ("k_max(alpha)/k_max(0)", "gamma_max(alpha)/gamma_max(0)",
                       "running exponents (implicit differentiation)")):
    ax.set_xlabel("1 - alpha^2"); ax.set_title(t); ax.legend(fontsize=7); ax.invert_xaxis()
plt.tight_layout()

In [ ]:
# cross-checks: 1b-ref's 10-ka peak fits (T.peak_fit through its table) and rung 1b's 5-ka ridge fits
tab = U.shear_tables()
kas_ref = sorted({k for k, _ in tab})
RIDGE = {}
for S in U.S_RIDGE:
    try:
        RIDGE[S] = {a: U.ridge_fit(rows) for a, rows in U.ridge_results(S).items() if len(rows) == 5}
    except Exception as e:
        print("ridge read failed", S, e)
print("   S     alpha | Newton k*      g*/g*(0)   | 1b-ref peak fit k (+-)      g/g(0) | 1b ridge fit k (+-)        "
      "| (1-a^2)^(4/7)")
for S in R.S_KA:
    if 0.0 not in KA[S]:
        continue
    g00 = KA[S][0.0]["gmax"]
    pk0 = T.peak_fit(kas_ref, [complex(tab[(k, 0.0)][S]["value"]).real for k in kas_ref]) if S in (1e3, 1e4, 1e5) else None
    for al, r in sorted(KA[S].items()):
        s = f"  {S:.0e}  {al:5.3f} | {r['kmax']:.6f}  {r['gmax']/g00:.5f}   |"
        if pk0 is not None and all((k, al) in tab for k in kas_ref):
            pk = T.peak_fit(kas_ref, [complex(tab[(k, al)][S]["value"]).real for k in kas_ref])
            s += f" {pk[0]:.4f} ({pk[2]:.0e})  {pk[1]/pk0[1]:.5f} |"
        else:
            s += " "*33 + "|"
        rf = RIDGE.get(S, {}).get(al)
        s += f" {rf['kmax']:.4f} ({rf['kmax_err']:.0e}) {'in ' if rf['inside'] else 'OUT'} |" if rf else " "*25 + "|"
        print(s + f" {(1-al*al)**(4/7):.4f}")
print("\nrung 1b-ref's quoted gamma_max(alpha)/gamma_max(0) = 0.955, 0.80, 0.58, 0.28 at alpha = 0.3, 0.6, 0.8, 0.95")

**Result (section 3).** *To be written from the full campaign.*

### 4. $k_{max}(S)$ and $\gamma_{max}(S)$ at fixed $\alpha$ (the paper's fig. 4), and the running slopes

Newton at $S=10^3\ldots10^5$ ($10^6$ at $\alpha=0$ if the optional phase ran), continued in $S$ with the previous solution's
implicit slopes. The **running** exponents $d\ln k^*/d\ln S=\eta\gamma_{k\eta}/(k^*\gamma_{kk})$ and
$d\ln\gamma^*/d\ln S=-(\eta/\gamma^*)\partial_\eta\gamma$ come from each solution alone; they are checked against the
secant slopes between neighbouring $S$, and at $\alpha=0$ against the exact relation's running slopes. The paper's
asymptotes are $-1/7$ and $-3/7$ at every $\alpha$ (section 0); the shear regime also needs $\delta_{in}/a\ll1$, printed.

In [ ]:
print("  alpha    S    | k*          gamma*         | d ln k*/d ln S: implicit  secant  theory(a=0) | "
      "d ln g*/d ln S: implicit  secant  theory | delta_in/a  grid dk/k")
for al in R.ALPHA_KS:
    Ss = sorted(KS[al])
    for i, S in enumerate(Ss):
        r = KS[al][S]
        sec_k = sec_g = np.nan
        if 0 < i < len(Ss) - 1:
            a_, b_ = KS[al][Ss[i - 1]], KS[al][Ss[i + 1]]
            L = np.log(Ss[i + 1]/Ss[i - 1])
            sec_k, sec_g = np.log(b_["kmax"]/a_["kmax"])/L, np.log(b_["gmax"]/a_["gmax"])/L
        tk, tg = R.theory_kmax_slopes(S) if al == 0 else (np.nan, np.nan)
        d = R.delta_in(S, r["kmax"], al) if al > 0 else np.nan
        print(f"  {al:5.3f}  {S:.0e} | {r['kmax']:.8f}  {r['gmax']:.8e} |        {r['slope_k_S']:+.5f}   {sec_k:+.5f}  "
              f"{tk:+.5f}    |          {r['slope_g_S']:+.5f}   {sec_g:+.5f}  {tg:+.5f} |  {d:.3f}     "
              f"{r['grid_dk']/r['kmax']:+.1e}")

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(17, 4.5))
for j, al in enumerate(R.ALPHA_KS):
    Ss = np.array(sorted(KS[al]))
    if not len(Ss):
        continue
    axs[0].loglog(Ss, [KS[al][S]["kmax"] for S in Ss], "o-", ms=4, color=f"C{j}", label=f"alpha = {al}")
    axs[1].loglog(Ss, [KS[al][S]["gmax"] for S in Ss], "o-", ms=4, color=f"C{j}", label=f"alpha = {al}")
    axs[2].semilogx(Ss, [KS[al][S]["slope_k_S"] for S in Ss], "o-", ms=4, color=f"C{j}", label=f"k*, alpha = {al}")
    axs[2].semilogx(Ss, [KS[al][S]["slope_g_S"] for S in Ss], "s--", ms=4, color=f"C{j}", label=f"gamma*, alpha = {al}")
SS = np.geomspace(1e3, 1e6, 40)
th = np.array([R.theory_kmax(S) for S in SS])
axs[0].loglog(SS, th[:, 0], "k-", lw=0.8, label="theory alpha = 0")
axs[1].loglog(SS, th[:, 1], "k-", lw=0.8, label="theory alpha = 0")
axs[2].semilogx(SS, [R.theory_kmax_slopes(S)[0] for S in SS], "k-", lw=0.8, label="theory (alpha = 0)")
axs[2].semilogx(SS, [R.theory_kmax_slopes(S)[1] for S in SS], "k-", lw=0.8)
for y in (-1/7, -3/7):
    axs[2].axhline(y, color="k", ls=":", lw=0.8)
for ax, t in zip(axs, ("k_max(S)", "gamma_max(S)", "running exponents d ln k*/d ln S, d ln gamma*/d ln S")):
    ax.set_xlabel("S"); ax.set_title(t); ax.legend(fontsize=7)
plt.tight_layout()

In [ ]:
# alpha = 0: Newton against the exact relation (finite-S correction), and the 2 nx re-solves
if 0.0 in KS:
    print("alpha = 0:   S   | k*/k_th - 1     gamma*/gamma_th")
    for S, r in sorted(KS[0.0].items()):
        kth, gth = R.theory_kmax(S)
        print(f"          {S:.0e} |  {r['kmax']/kth - 1:+.4e}     {r['gmax']/gth:.6f}")
print("\nnx doubling of k*: the kmax_alpha solution at nx against a full Newton at 2 nx")
for (S, al), r2 in G2.items():
    r1 = KA.get(S, {}).get(al)
    if r1:
        print(f"  S={S:.0e} alpha={al}: nx {r1['nx']} -> {r2['nx']}: d k*/k* = {r2['kmax']/r1['kmax'] - 1:+.2e}, "
              f"d gamma*/gamma* = {r2['gmax']/r1['gmax'] - 1:+.2e}; the nx/2 estimate was {r1['grid_dk']/r1['kmax']:+.1e}")

**Result (section 4).** *To be written from the full campaign.*

### 5. Summary

*To be written from the full campaign.*